# 04 — Model experiments (not yet part of the baseline)

**Selection rule, fixed before any run:** feature sets and parameters are compared only on the validation folds (mean AUC-PR over five rounds). A change is accepted only if its mean gain over the current choice is larger than the standard deviation of the per-round gains. Test folds are looked at only once, for the final choice, in section 5.

The baseline test score of notebook 03 was already seen before this notebook was written. Its feature set stays the reference; nothing here is chosen by test score.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.features import FEATURES, WINDOW_END, account_features
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    SEED,
    cached_folds,
    cv_predict,
    fold_roles,
    split_groups,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)
folds = cached_folds(labels, groups, DATA_INTERIM)

X_base = account_features(df)
y = labels.loc[X_base.index]
fold_of = folds.loc[X_base.index]
print(f"model rows: {len(X_base):,}  positives: {y.sum():,}")

model rows: 515,078  positives: 6,347


## 1. Where does the single currency change come from?

**Question:** notebook 03 shows that accounts with exactly one currency change are often positive, although no laundering transaction changes currency, and that this change is almost always an ACH self-transaction at about the time of the first laundering. Is it paired with a laundering transaction, in the same minute and with the same currency and amount?

In [2]:
early = df[df["timestamp"] < WINDOW_END].copy()
early["sender"] = account_key(early["from_bank"], early["from_account"])
early["receiver"] = account_key(early["to_bank"], early["to_account"])
is_self = early["sender"] == early["receiver"]
changes = early["receiving_currency"] != early["payment_currency"]

conversions = early.loc[is_self & changes]
print(f"currency-changing transactions: {changes.sum():,}")
print(f"  self-transactions: {(is_self & changes).sum():,}")
print(f"  labelled laundering: {early.loc[changes, 'is_laundering'].sum()}")

sends = early.loc[(early["is_laundering"] == 1) & ~is_self]
paired = conversions.merge(
    sends,
    left_on=["sender", "timestamp", "receiving_currency"],
    right_on=["sender", "timestamp", "payment_currency"],
    suffixes=("_conv", "_send"),
)
same_amount = np.isclose(
    paired["amount_received_conv"], paired["amount_paid_send"], rtol=0.01
)
n_paired = paired.drop_duplicates(["sender", "timestamp"]).shape[0]
print(f"self conversions: {len(conversions):,}")
print(f"  same minute, laundering send in that currency: {n_paired:,}")
print(f"  among those pairs, amounts within 1%: {same_amount.mean():.1%}")
print(f"laundering sends: {len(sends):,}")

currency-changing transactions: 71,779
  self-transactions: 69,589
  labelled laundering: 0
self conversions: 69,589
  same minute, laundering send in that currency: 1,619
  among those pairs, amounts within 1%: 93.5%
laundering sends: 4,514


In [3]:
# Control: is this pairing special to laundering, or do clean sends have it too?
# Counted per conversion row: an account can convert twice in one minute.
clean_sends = early.loc[(early["is_laundering"] == 0) & ~is_self]
clean_paired = conversions.merge(
    clean_sends,
    left_on=["sender", "timestamp", "receiving_currency"],
    right_on=["sender", "timestamp", "payment_currency"],
    suffixes=("_conv", "_send"),
)
clean_same = np.isclose(
    clean_paired["amount_received_conv"], clean_paired["amount_paid_send"], rtol=0.01
)
conversion_keys = pd.MultiIndex.from_frame(
    conversions[["sender", "timestamp", "receiving_currency"]]
)


def matches(subset):
    columns = ["sender", "timestamp", "payment_currency"]
    return conversion_keys.isin(pd.MultiIndex.from_frame(subset[columns]))


has_laundering, has_clean = matches(sends), matches(clean_sends)
has_any = has_laundering | has_clean
print(f"conversions with a same-minute send: {has_any.sum():,} of {len(conversions):,}")
print(f"  with a clean send: {has_clean.sum():,}")
print(f"  with a laundering send: {has_laundering.sum():,}")
print(f"  with both: {(has_clean & has_laundering).sum():,}")
print(f"  with neither: {(~has_any).sum():,}")
print(f"clean pairs, amounts within 1%: {clean_same.mean():.1%}")
print(
    f"share of matched conversions with a laundering send: "
    f"{has_laundering.sum() / has_any.sum():.2%}"
)
overall_share = len(sends) / (~is_self).sum()
print(f"laundering share of non-self sends: {overall_share:.3%}")
ach_sends = clean_sends[clean_sends["payment_format"] == "ACH"]
ach_bad = sends[sends["payment_format"] == "ACH"]
ach_share = len(ach_bad) / (len(ach_bad) + len(ach_sends))
print(f"laundering share of non-self ACH sends: {ach_share:.3%}")
print("format of laundering sends matched to a conversion:")
print(paired["payment_format_send"].value_counts().to_string())

conversions with a same-minute send: 69,516 of 69,589
  with a clean send: 68,018
  with a laundering send: 1,619
  with both: 121
  with neither: 73
clean pairs, amounts within 1%: 95.3%
share of matched conversions with a laundering send: 2.33%
laundering share of non-self sends: 0.101%
laundering share of non-self ACH sends: 0.722%
format of laundering sends matched to a conversion:
payment_format_send
ACH    1619


## 2. More features

Three candidate groups, all computed from the 1–10 September window, without labels, without account identities as values, and without absolute dates:

- **amounts:** sums, medians and maxima in US dollars, and how balanced in and out are. Self-transactions are left out of the flows. The data has no exchange-rate table, so one rate per currency is taken from the transactions themselves (median of received/paid against US dollars). This uses no labels, but it is computed once on the whole window.
- **timing:** the busiest day and hour of an account, and the median gap between its transactions.
- **network:** reciprocal counterparties, how connected the counterparties are, counterparty banks, currencies, and ACH counterparties. Only counts; no counterparty identity or label.

In [4]:
%%time
# One exchange rate per currency, to US dollars, from the transactions themselves.
USD = "US Dollar"
pairs_fx = early.loc[changes]
to_usd = pairs_fx[pairs_fx["receiving_currency"] == USD]
from_usd = pairs_fx[pairs_fx["payment_currency"] == USD]
direct = (
    (to_usd["amount_received"] / to_usd["amount_paid"])
    .groupby(to_usd["payment_currency"])
    .median()
)
inverse = (
    (from_usd["amount_paid"] / from_usd["amount_received"])
    .groupby(from_usd["receiving_currency"])
    .median()
)
rates = direct.combine_first(inverse)
rates[USD] = 1.0
currencies = set(early["payment_currency"]) | set(early["receiving_currency"])
assert currencies <= set(rates.index), "a currency without a rate"
print(f"currencies: {len(currencies)}  with a rate: {rates.size}")

flows = early.loc[~is_self].assign(
    paid_usd=lambda t: t["amount_paid"] * t["payment_currency"].map(rates),
    received_usd=lambda t: t["amount_received"] * t["receiving_currency"].map(rates),
)
index = X_base.index


def aligned(series, fill=0.0):
    return series.reindex(index, fill_value=fill).astype("float64")


sent = flows.groupby("sender")["paid_usd"]
received = flows.groupby("receiver")["received_usd"]
sent_sum, received_sum = aligned(sent.sum()), aligned(received.sum())
amounts = pd.DataFrame(
    {
        "log_sent_usd": np.log1p(sent_sum),
        "log_received_usd": np.log1p(received_sum),
        "log_median_sent_usd": np.log1p(aligned(sent.median())),
        "log_median_received_usd": np.log1p(aligned(received.median())),
        "log_max_sent_usd": np.log1p(aligned(sent.max())),
        "log_max_received_usd": np.log1p(aligned(received.max())),
        "flow_balance": (received_sum - sent_sum) / (received_sum + sent_sum + 1.0),
    },
    index=index,
)

touches = pd.concat(
    [
        early[["sender", "timestamp"]].rename(columns={"sender": "account"}),
        early.loc[~is_self, ["receiver", "timestamp"]].rename(
            columns={"receiver": "account"}
        ),
    ]
).sort_values(["account", "timestamp"])
per_day = touches.groupby(["account", touches["timestamp"].dt.floor("D")]).size()
per_hour = touches.groupby(["account", touches["timestamp"].dt.floor("h")]).size()
gaps = touches.groupby("account")["timestamp"].diff().dt.total_seconds() / 3600
timing = pd.DataFrame(
    {
        "max_tx_per_day": aligned(per_day.groupby(level=0).max()),
        "max_tx_per_hour": aligned(per_hour.groupby(level=0).max()),
        "median_gap_hours": gaps.groupby(touches["account"]).median().reindex(index),
    },
    index=index,
)

pairs = flows[["sender", "receiver"]].drop_duplicates()
out_degree, in_degree = pairs["sender"].value_counts(), pairs["receiver"].value_counts()
reversed_pairs = pairs.rename(columns={"sender": "receiver", "receiver": "sender"})
reciprocal = pairs.merge(reversed_pairs, on=["sender", "receiver"])
ach = flows[flows["payment_format"] == "ACH"][["sender", "receiver"]].drop_duplicates()
currencies_used = pd.concat(
    [
        early[["sender", "payment_currency"]].set_axis(["account", "currency"], axis=1),
        early[["receiver", "receiving_currency"]].set_axis(
            ["account", "currency"], axis=1
        ),
    ]
).drop_duplicates()
bank_of = lambda keys: keys.str.split(":", n=1).str[0]  # noqa: E731
network = pd.DataFrame(
    {
        "reciprocal_counterparties": aligned(reciprocal["sender"].value_counts()),
        "mean_receiver_in_degree": aligned(
            pairs["receiver"].map(in_degree).groupby(pairs["sender"]).mean()
        ),
        "mean_sender_out_degree": aligned(
            pairs["sender"].map(out_degree).groupby(pairs["receiver"]).mean()
        ),
        "ach_receivers": aligned(ach["sender"].value_counts()),
        "ach_senders": aligned(ach["receiver"].value_counts()),
        "receiver_banks": aligned(
            bank_of(pairs["receiver"]).groupby(pairs["sender"]).nunique()
        ),
        "sender_banks": aligned(
            bank_of(pairs["sender"]).groupby(pairs["receiver"]).nunique()
        ),
        "n_currencies": aligned(currencies_used["account"].value_counts()),
    },
    index=index,
)

X_all = pd.concat([X_base, amounts, timing, network], axis=1)
BLOCKS = {
    "amounts": list(amounts.columns),
    "timing": list(timing.columns),
    "network": list(network.columns),
}
assert not X_all.columns.duplicated().any()
assert "is_laundering" not in X_all.columns
print(f"features: {X_all.shape[1]}")
print("missing values:", X_all.isna().sum().loc[lambda s: s > 0].to_dict())

currencies: 15  with a rate: 15


features: 33
missing values: {'median_gap_hours': 90862}
CPU times: total: 27.7 s
Wall time: 27.7 s


## 3. Feature groups, compared on validation folds only

In [5]:
%%time
BASE_PARAMS = {
    "n_estimators": 1000,
    "learning_rate": 0.1,
    "max_depth": 6,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "early_stopping_rounds": 50,
    "random_state": SEED,
}


def run(columns, params=None, target=None, threshold=None):
    """Five rounds through cv_predict. Validation AUC-PR is recorded in every round."""
    target = y if target is None else target
    settings = {**BASE_PARAMS, **(params or {})}
    val_scores, models = [], []

    def fit(X_train, y_train, X_val, y_val):
        model = xgb.XGBClassifier(**settings)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        val_scores.append(
            average_precision_score(y_val, model.predict_proba(X_val)[:, 1])
        )
        models.append(model)
        return lambda rows: model.predict_proba(rows)[:, 1]

    result = cv_predict(X_all[columns], target, fold_of, fit, threshold)
    return {"val": pd.Series(val_scores), "result": result, "models": models}


def compare(candidate, current):
    gain = candidate["val"] - current["val"]
    return {
        "val_auc_pr": candidate["val"].mean(),
        "val_sd": candidate["val"].std(ddof=1),
        "gain": gain.mean(),
        "gain_sd": gain.std(ddof=1),
        "accepted": gain.mean() > gain.std(ddof=1),
    }


reference = run(FEATURES)
feature_sets = {
    "+ amounts": FEATURES + BLOCKS["amounts"],
    "+ timing": FEATURES + BLOCKS["timing"],
    "+ network": FEATURES + BLOCKS["network"],
    "+ all three": list(X_all.columns),
}
runs = {name: run(columns) for name, columns in feature_sets.items()}
table = {"base 15": compare(reference, reference)}
table |= {name: compare(found, reference) for name, found in runs.items()}
feature_table = pd.DataFrame(table).T
feature_table

CPU times: total: 28min 46s
Wall time: 1min 58s


,val_auc_pr,val_sd,gain,gain_sd,accepted
base 15,0.291986,0.02222,0.0,0.0,False
+ amounts,0.32176,0.022559,0.029774,0.004299,True
+ timing,0.301579,0.020475,0.009592,0.002286,True
+ network,0.336514,0.014634,0.044528,0.013655,True
+ all three,0.35417,0.014829,0.062183,0.013969,True


In [6]:
# Selection: the accepted feature set with the highest validation AUC-PR.
accepted = feature_table[feature_table["accepted"].astype(bool)]
if len(accepted):
    chosen_set = accepted["val_auc_pr"].astype(float).idxmax()
    chosen_columns, chosen_run = feature_sets[chosen_set], runs[chosen_set]
else:
    chosen_set, chosen_columns, chosen_run = "base 15", list(FEATURES), reference
print(f"chosen feature set: {chosen_set} ({len(chosen_columns)} features)")

chosen feature set: + all three (33 features)


## 4. Parameters: a small random search, on validation folds only

In [7]:
%%time
rng = np.random.default_rng(SEED)
SPACE = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5, 20],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [1, 5],
}
N_TRIALS = 12
trials = []
for _trial in range(N_TRIALS):
    params = {name: rng.choice(values).item() for name, values in SPACE.items()}
    params["n_estimators"] = 3000
    found = run(chosen_columns, params)
    trials.append({**compare(found, chosen_run), **params, "run": found})
search = pd.DataFrame(trials).sort_values("val_auc_pr", ascending=False)
search.drop(columns="run")

CPU times: total: 2h 51min 12s
Wall time: 11min 28s


,val_auc_pr,val_sd,gain,gain_sd,accepted,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,reg_lambda,n_estimators
5,0.363660,0.016340,0.009491,0.002825,True,6,0.03,1,0.9,1.0,1,3000
9,0.361528,0.013877,0.007358,0.003031,True,6,0.03,5,0.7,1.0,5,3000
1,0.360812,0.015744,0.006642,0.003815,True,4,0.10,1,0.7,0.8,5,3000
4,0.360363,0.014416,0.006193,0.002009,True,10,0.05,5,1.0,0.8,1,3000
0,0.360355,0.016282,0.006186,0.003087,True,4,0.10,5,0.9,0.8,5,3000
3,0.359393,0.014373,0.005223,0.003133,True,10,0.05,5,0.9,0.6,5,3000
7,0.358644,0.016066,0.004474,0.003880,True,8,0.05,1,1.0,0.8,5,3000
2,0.358584,0.012898,0.004415,0.002829,True,8,0.10,20,1.0,0.8,1,3000
8,0.356260,0.013566,0.002090,0.001595,True,8,0.10,20,0.7,0.8,1,3000
11,0.354325,0.020347,0.000156,0.008318,False,10,0.05,1,0.9,1.0,1,3000


In [8]:
# Selection: the accepted trial with the highest validation AUC-PR, or the defaults.
accepted_trials = search[search["accepted"].astype(bool)]
final_params, final_run, tuned = None, chosen_run, False
if len(accepted_trials):
    best = accepted_trials.iloc[0]
    final_params = {name: best[name] for name in [*SPACE, "n_estimators"]}
    final_params = {
        name: (value.item() if hasattr(value, "item") else value)
        for name, value in final_params.items()
    }
    final_run, tuned = best["run"], True
print("tuned parameters accepted:", tuned)
print("final parameters:", final_params or "defaults of the baseline")
print(f"final validation AUC-PR: {final_run['val'].mean():.4f}")

tuned parameters accepted: True
final parameters: {'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 1, 'subsample': 0.9, 'colsample_bytree': 1.0, 'reg_lambda': 1, 'n_estimators': 3000}
final validation AUC-PR: 0.3637


## 5. The final choice on the test folds (looked at once)

In [9]:
def test_scores(found, target=None):
    target = y if target is None else target
    scores = found["result"].scores
    rows = []
    for k in range(N_FOLDS):
        test = fold_of == k
        rows.append(
            {
                "auc_pr": average_precision_score(target[test], scores[test]),
                "auc_roc": roc_auc_score(target[test], scores[test]),
            }
        )
    return pd.DataFrame(rows)


base_test, final_test = test_scores(reference), test_scores(final_run)
paired = pd.DataFrame(
    {
        "baseline": base_test["auc_pr"],
        "final": final_test["auc_pr"],
        "gain": final_test["auc_pr"] - base_test["auc_pr"],
        "best_iteration": [m.best_iteration for m in final_run["models"]],
    }
)
print(paired.round(4).to_string())


def mean_sd(values):
    return f"{values.mean():.4f} +/- {values.std(ddof=1):.4f}"


print("\nbaseline AUC-PR:", mean_sd(base_test["auc_pr"]))
print("final AUC-PR:", mean_sd(final_test["auc_pr"]))
print("final AUC-ROC:", mean_sd(final_test["auc_roc"]))
print("validation AUC-PR of the final choice:", mean_sd(final_run["val"]))
print("gain on test, per round:", mean_sd(paired["gain"]))

   baseline   final    gain  best_iteration
0    0.3015  0.3812  0.0797             282
1    0.2507  0.3451  0.0944             344
2    0.2962  0.3552  0.0590             435
3    0.2954  0.3601  0.0647             565
4    0.3027  0.3777  0.0750             605

baseline AUC-PR: 0.2893 +/- 0.0218
final AUC-PR: 0.3639 +/- 0.0153
final AUC-ROC: 0.9109 +/- 0.0072
validation AUC-PR of the final choice: 0.3637 +/- 0.0163
gain on test, per round: 0.0746 +/- 0.0138


In [10]:
%%time
# Checks on the final choice: shuffled labels, and without the suspect features.
shuffled = pd.Series(
    y.sample(frac=1, random_state=0).to_numpy(), index=y.index, name=y.name
)
print(
    "AUC-PR with shuffled labels:",
    mean_sd(
        test_scores(run(chosen_columns, final_params, target=shuffled), shuffled)[
            "auc_pr"
        ]
    ),
)
suspects = {
    "without currency_change_tx": {"currency_change_tx"},
    "without ACH features": {"ach_out_tx", "ach_in_tx", "ach_receivers", "ach_senders"},
    "without both": {
        "currency_change_tx",
        "ach_out_tx",
        "ach_in_tx",
        "ach_receivers",
        "ach_senders",
    },
}
for name, dropped in suspects.items():
    kept = [column for column in chosen_columns if column not in dropped]
    print(f"{name}:", mean_sd(test_scores(run(kept, final_params))["auc_pr"]))

AUC-PR with shuffled labels: 0.0124 +/- 0.0003


without currency_change_tx: 0.3586 +/- 0.0174


without ACH features: 0.3275 +/- 0.0269


without both: 0.3205 +/- 0.0279
CPU times: total: 1h 14min 25s
Wall time: 4min 58s


In [11]:
# How the score looks in use: threshold from the validation fold (best F1 there).
def best_f1_threshold(y_val, val_scores):
    precision, recall, thresholds = precision_recall_curve(y_val, val_scores)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return thresholds[f1.argmax()]


flagged_run = run(chosen_columns, final_params, threshold=best_f1_threshold)
flags, scores = flagged_run["result"].flags, flagged_run["result"].scores
use = []
for k in range(N_FOLDS):
    test = fold_of == k
    top = scores[test].nlargest(100).index
    use.append(
        {
            "flagged": flags[test].sum(),
            "precision": y[test][flags[test] == 1].mean(),
            "recall": flags[test][y[test] == 1].mean(),
            "precision_top_100": y[top].mean(),
        }
    )
use = pd.DataFrame(use)
print(use.round(3).to_string())
print(use.mean().round(3).to_string())

ranks = final_run["result"].scores.rank(pct=True)
in_pattern = X_all.index.isin(members["account"])
print(f"\nmedian rank, pattern positives: {ranks[(y == 1) & in_pattern].median():.3f}")
print(f"median rank, other positives: {ranks[(y == 1) & ~in_pattern].median():.3f}")
print(f"median rank, negatives: {ranks[y == 0].median():.3f}")

   flagged  precision  recall  precision_top_100
0      693      0.561   0.306               0.99
1      404      0.720   0.229               1.00
2      406      0.778   0.249               1.00
3      430      0.767   0.260               1.00
4      450      0.749   0.265               1.00
flagged              476.600
precision              0.715
recall                 0.262
precision_top_100      0.998

median rank, pattern positives: 0.995
median rank, other positives: 0.928
median rank, negatives: 0.494


## 6. SHAP

TreeSHAP through XGBoost (`pred_contribs`), so no new dependency. Every account is explained by the model that never saw it: the model of the round in which the account was in the test fold.

In [12]:
%%time
contributions = []
for k, model in enumerate(final_run["models"]):
    assert (fold_roles(fold_of, k) == "test").eq(fold_of == k).all()
    rows = X_all.loc[fold_of == k, chosen_columns]
    matrix = xgb.DMatrix(rows)
    limit = (0, model.best_iteration + 1)
    values = model.get_booster().predict(
        matrix, pred_contribs=True, iteration_range=limit
    )
    margin = model.get_booster().predict(
        matrix, output_margin=True, iteration_range=limit
    )
    assert np.allclose(values.sum(axis=1), margin, atol=1e-3), "SHAP does not add up"
    contributions.append(
        pd.DataFrame(values, index=rows.index, columns=[*chosen_columns, "base_value"])
    )
shap_values = pd.concat(contributions).loc[X_all.index]
only_features = shap_values[chosen_columns]

overall = pd.DataFrame(
    {
        "mean_abs_shap": only_features.abs().mean(),
        "mean_abs_shap_pos": only_features[y == 1].abs().mean(),
        "mean_shap_pos": only_features[y == 1].mean(),
        "mean_shap_neg": only_features[y == 0].mean(),
    }
).sort_values("mean_abs_shap", ascending=False)
overall.head(15).round(4)

CPU times: total: 25min 37s
Wall time: 1min 41s


,mean_abs_shap,mean_abs_shap_pos,mean_shap_pos,mean_shap_neg
ach_in_tx,0.5857,0.9035,0.6475,-0.2663
mean_sender_out_degree,0.3414,0.3250,0.1064,-0.1729
ach_receivers,0.3067,0.4112,0.1936,-0.1864
log_max_sent_usd,0.2904,0.2622,0.1151,-0.1411
in_tx,0.2705,0.0989,0.0531,-0.1964
log_max_received_usd,0.2342,0.2135,0.0338,-0.1721
median_gap_hours,0.2327,0.1148,0.0425,-0.1743
ach_senders,0.2141,0.3186,0.1056,-0.1484
ach_out_tx,0.2090,0.2605,0.1478,-0.1001
max_tx_per_day,0.2047,0.2270,0.0360,0.0902


In [13]:
# Two example cases, without account identities: the top true positive, and a
# positive the model ranks low. Margins are in log-odds.
def explain(account, title):
    row = only_features.loc[account]
    top = row.reindex(row.abs().sort_values(ascending=False).index).head(6)
    table = pd.DataFrame({"value": X_all.loc[account, top.index], "shap": top})
    score = final_run["result"].scores[account]
    print(f"{title}: score {score:.3f}, rank {ranks[account]:.4f}, label {y[account]}")
    print(table.round(3).to_string(), "\n")


final_scores = final_run["result"].scores
explain(final_scores[y == 1].idxmax(), "highest-scored positive")
low = final_scores[y == 1].sort_values().index[len(final_scores[y == 1]) // 10]
explain(low, "positive at the 10th percentile of positive scores")
explain(final_scores[y == 0].idxmax(), "highest-scored negative")

highest-scored positive: score 0.999, rank 1.0000, label 1
                    value   shap
n_currencies        2.000  1.760
ach_in_tx           2.000  1.620
ach_receivers       4.000  1.534
currency_change_tx  1.000  1.325
ach_out_tx          5.000  0.911
log_max_sent_usd    9.665  0.801 

positive at the 10th percentile of positive scores: score 0.008, rank 0.7431, label 1
                         value   shap
ach_in_tx                  0.0 -0.394
mean_sender_out_degree     0.0 -0.364
log_max_received_usd       0.0 -0.340
max_tx_per_day             1.0  0.301
ach_out_tx                 1.0  0.206
mean_receiver_in_degree   10.0 -0.154 

highest-scored negative: score 0.974, rank 0.9993, label 0
                    value   shap
n_currencies        2.000  2.310
currency_change_tx  1.000  1.509
ach_in_tx           2.000  1.276
ach_out_tx          2.000  0.649
log_max_sent_usd    8.877  0.630
ach_receivers       1.000  0.615 



## 7. Hyperparameter ablation and stability

Each parameter changes alone, around the final choice, on validation folds only. These tables inform the discussion; they do not change the final choice, and no further test score is used to choose.

In [14]:
# Keep the candidate features on disk (ignored by git) for an independent check.
X_all.to_parquet(DATA_INTERIM / "experiment_features.parquet")
rates.rename("rate_to_usd").to_frame().round(6)

,rate_to_usd
payment_currency,
Australian Dollar,0.707814
Bitcoin,11881.623158
Brazil Real,0.177101
Canadian Dollar,0.757978
Euro,1.171783
Mexican Peso,0.047297
Ruble,0.012853
Rupee,0.013616
Saudi Riyal,0.266588


In [15]:
%%time
# One parameter at a time, around the final choice. Validation folds only.
GRID = {
    "max_depth": [3, 4, 8, 10],
    "learning_rate": [0.05, 0.1, 0.2],
    "min_child_weight": [5, 20, 50],
    "subsample": [0.6, 1.0],
    "colsample_bytree": [0.5, 0.8],
    "reg_lambda": [5, 20],
    "scale_pos_weight": [10, 80],
}
ablation_rows = []
for name, values in GRID.items():
    for value in values:
        found = run(chosen_columns, {**final_params, name: value})
        ablation_rows.append(
            {
                "parameter": name,
                "value": value,
                **compare(found, final_run),
                "mean_best_iteration": np.mean(
                    [m.best_iteration for m in found["models"]]
                ),
                "max_best_iteration": max(m.best_iteration for m in found["models"]),
            }
        )
pd.DataFrame(ablation_rows).round(4)

CPU times: total: 7h 17min 25s
Wall time: 28min 56s


,parameter,value,val_auc_pr,val_sd,gain,gain_sd,accepted,mean_best_iteration,max_best_iteration
0,max_depth,3.00,0.3576,0.0171,-0.0061,0.0013,False,1420.2,1523
1,max_depth,4.00,0.3626,0.0150,-0.0011,0.0025,False,1092.4,1769
2,max_depth,8.00,0.3619,0.0150,-0.0018,0.0025,False,288.6,357
3,max_depth,10.00,0.3570,0.0190,-0.0066,0.0047,False,225.2,293
4,learning_rate,0.05,0.3629,0.0171,-0.0008,0.0024,False,313.4,405
5,learning_rate,0.10,0.3565,0.0143,-0.0071,0.0026,False,182.6,266
6,learning_rate,0.20,0.3467,0.0127,-0.0169,0.0068,False,94.2,144
7,min_child_weight,5.00,0.3620,0.0135,-0.0016,0.0038,False,445.6,581
8,min_child_weight,20.00,0.3605,0.0139,-0.0031,0.0036,False,654.0,876
9,min_child_weight,50.00,0.3579,0.0155,-0.0058,0.0026,False,944.6,1202


In [16]:
%%time
# Remove each feature block from the final choice, on validation folds.
block_rows = {}
for name, columns in {**BLOCKS, "base counts": list(FEATURES)}.items():
    kept = [column for column in chosen_columns if column not in columns]
    block_rows[f"without {name}"] = compare(run(kept, final_params), final_run)
pd.DataFrame(block_rows).T.drop(columns="accepted")

CPU times: total: 1h 5min 11s
Wall time: 4min 19s


,val_auc_pr,val_sd,gain,gain_sd
without amounts,0.345406,0.014772,-0.018254,0.002493
without timing,0.359494,0.014685,-0.004166,0.002006
without network,0.33436,0.02114,-0.0293,0.009165
without base counts,0.328862,0.020423,-0.034798,0.005067


In [17]:
%%time
# Stability of the final choice: the same run twice, and other random seeds.
again = run(chosen_columns, final_params)
same = np.array_equal(again["result"].scores, final_run["result"].scores)
print("same seed gives identical scores:", same)
largest = (again["result"].scores - final_run["result"].scores).abs().max()
print("largest score difference:", largest)

seed_rows = []
for seed in [1, 2, 3]:
    found = run(chosen_columns, {**final_params, "random_state": seed})
    seed_rows.append(
        {
            "seed": seed,
            "val_auc_pr": found["val"].mean(),
            "test_auc_pr": test_scores(found)["auc_pr"].mean(),
        }
    )
print(pd.DataFrame(seed_rows).round(4).to_string(index=False))

same seed gives identical scores: True
largest score difference: 0.0


 seed  val_auc_pr  test_auc_pr
    1      0.3639       0.3644
    2      0.3646       0.3640
    3      0.3637       0.3649
CPU times: total: 1h 30min 32s
Wall time: 6min 1s


## 8. Two gaps in the checks above

These tables do not change the final choice.

- Section 5 removes `currency_change_tx` but keeps `n_currencies`, which can carry the same signal. Here both are removed.
- Section 7 changes only the model seed. Here the fold assignment changes too. The parameters were chosen on the original folds, so this also shows how much the score depends on that one split.

In [18]:
# Positive rate by number of currencies, and the final choice without this signal.
by_currencies = pd.DataFrame(
    {"accounts": y.groupby(X_all["n_currencies"]).size()}
).assign(
    positives=y.groupby(X_all["n_currencies"]).sum(),
    positive_rate=y.groupby(X_all["n_currencies"]).mean(),
)
print(by_currencies.head(6).round(4).to_string())

CURRENCY = {"currency_change_tx", "n_currencies"}
ACH = {"ach_out_tx", "ach_in_tx", "ach_receivers", "ach_senders"}
for name, dropped in {
    "without both currency features": CURRENCY,
    "without both currency features and ACH features": CURRENCY | ACH,
}.items():
    kept = [column for column in chosen_columns if column not in dropped]
    found = run(kept, final_params)
    print(f"\n{name} ({len(kept)} features)")
    print("  validation:", mean_sd(found["val"]))
    print("  test:", mean_sd(test_scores(found)["auc_pr"]))

              accounts  positives  positive_rate
n_currencies                                    
1.0             506013       5069         0.0100
2.0               8741       1198         0.1371
3.0                273         36         0.1319
4.0                 18         12         0.6667
5.0                 10         10         1.0000
6.0                  8          8         1.0000



without both currency features (31 features)
  validation: 0.3576 +/- 0.0174


  test: 0.3573 +/- 0.0168



without both currency features and ACH features (27 features)
  validation: 0.3225 +/- 0.0303


  test: 0.3224 +/- 0.0275


In [19]:
%%time
# Other fold assignments: baseline and final choice on the same new folds.
original_folds = fold_of
fold_rows = []
for fold_seed in [1, 2]:
    fold_of = cached_folds(labels, groups, DATA_INTERIM, seed=fold_seed).loc[
        X_base.index
    ]
    base_found = test_scores(run(FEATURES))["auc_pr"]
    final_found = test_scores(run(chosen_columns, final_params))["auc_pr"]
    gain = final_found - base_found
    fold_rows.append(
        {
            "fold_seed": fold_seed,
            "baseline": mean_sd(base_found),
            "final": mean_sd(final_found),
            "gain": mean_sd(gain),
        }
    )
fold_of = original_folds
print(pd.DataFrame(fold_rows).to_string(index=False))

 fold_seed          baseline             final              gain
         1 0.2861 +/- 0.0131 0.3603 +/- 0.0146 0.0742 +/- 0.0117
         2 0.2927 +/- 0.0166 0.3634 +/- 0.0154 0.0707 +/- 0.0168
CPU times: total: 53min 12s
Wall time: 4min 27s
